# Colab Demo: Tiny GRPO on the SME Liquidity Environment

This notebook stays fully in process. It does not use `server.app`; it uses `SMELiquidityEnvironment`, `rl/bridge.py`, and `rl/demo.py` directly.


## 1. Clone and install

Recommended runtime: GPU if available. A T4 is enough for a tiny demo run.


In [ ]:
!git clone https://github.com/<your-username>/OpenEnv_SME_Negotiator.git
%cd OpenEnv_SME_Negotiator
%pip install -e ".[rl]"


## 2. Smoke test the liquidity environment


In [ ]:
from server.environment import SMELiquidityEnvironment

env = SMELiquidityEnvironment(total_periods=2)
observation = env.reset(seed=42, difficulty="hard", task_name="liquidity-correlation-hard")
print(observation)


## 3. Baseline heuristic episodes

Use the deterministic notebook helper to get pre-training rewards and a readable trajectory.


In [ ]:
from rl.demo import run_heuristic_episode

baseline_runs = [run_heuristic_episode(seed=i, total_periods=2, task_name="liquidity-correlation-hard") for i in range(5)]
baseline_rewards = [item["total_reward"] for item in baseline_runs]
print("Baseline rewards:", baseline_rewards)
print("\nSample baseline transcript:\n")
print(baseline_runs[0]["transcript"])


## 4. Tiny GRPO training run

This keeps the same default model family as the main training scripts, but with a tiny step budget for demo purposes.


In [ ]:
from rl.demo import demo_train_grpo

history = demo_train_grpo(
    model_name="Qwen/Qwen2.5-1.5B-Instruct",
    steps=10,
    total_periods=2,
    task_name="liquidity-correlation-hard",
    num_samples=8,
    output_dir="outputs/colab_demo_grpo",
)
history


## 5. Plot reward vs. step


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(history["steps"], history["avg_reward"], marker="o")
plt.xlabel("Training step")
plt.ylabel("Average episode reward")
plt.title("Tiny GRPO Demo on SMELiquidityEnvironment")
plt.grid(alpha=0.3)
plt.show()


## 6. Before / after trajectory comparison


In [ ]:
from rl.demo import run_policy_episode

print("=== Before Training (Heuristic) ===")
print(run_policy_episode(policy="heuristic", seed=123, total_periods=2, task_name="liquidity-correlation-hard"))

print("\n=== After Tiny GRPO Demo ===")
print(run_policy_episode(policy="trained", seed=123, total_periods=2, task_name="liquidity-correlation-hard", checkpoint_path=history["checkpoint_path"]))


## 7. Troubleshooting

- If the training cell is slow, reduce `steps` and `num_samples`.
- If Colab RAM is tight, restart the runtime before the GRPO cell.
- If you want the richer Stage 6 path later, look at `rl/train_grpo_trl.py` and `rl/train_grpo_unsloth.py` for self-play, curriculum, and rubric flags.
